# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

fatal: destination path 'SQuAD-explorer' already exists and is not an empty directory.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [4]:
type(train_data)

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"])

list

In [7]:
len(train_data["data"])

442

In [8]:
type(train_data["data"][0])

dict

In [9]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"])

55

In [12]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [13]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [14]:
import pandas as pd

# extract each document
documents = []
for i in train_data['data']:
    documents.append(i)

# extract each document's title, paragraph indices, and paragraph contexts
titles            = []
paragraph_idx     = []
paragraph_context = []
for i in documents:

    # get each document title
    titles.append(i['title'])

    # paragraph count + indices
    n_paragraphs = len(i['paragraphs'])
    p_indices = list(range(0, n_paragraphs))
    paragraph_idx.append(p_indices)

    # get each paragraph indice's context
    doc_context = []    # joined list of contexts for each paragraph per document
    for p in p_indices:
        context = i['paragraphs'][p]['context']
        doc_context.append(context)

    # append all contexts for document's paragraphs to a singular index
    paragraph_context.append(doc_context)

if (len(titles) != len(paragraph_idx)) and (len(paragraph_idx) != len(paragraph_context)):
    print(f'Incorrect parsing:      titles: {len(titles)}, paragraph_index: {len(paragraph_idx)}, paragraph_context: {len(paragraph_context)}')



data = []
for t, indices, context in zip(titles, paragraph_idx, paragraph_context):
    
    # loop through nested lists of indices/context per document
    for idx, ctx in zip(indices, context):
        # append one sample per paragraph within the document
        data.append({
            'document_title': t,
            'paragraph_index': idx,
            'paragraph_context': ctx
        })

parsed_df = pd.DataFrame(data)
parsed_df.sample(25)

,document_title,paragraph_index,paragraph_context
2180,Human_Development_Index,5,The 2015 Human Development Report by the Unite...
15930,Vacuum,8,But although it meets the definition of outer ...
9780,Thuringia,1,The name Thuringia or Thüringen derives from t...
11660,History_of_science,64,Humboldtian science refers to the early 19th c...
6397,Old_English,5,The first example is taken from the opening li...
10868,Pope_Paul_VI,54,Paul VI did renounce many traditional symbols ...
18013,Egypt,46,The Suez Canal is an artificial sea-level wate...
700,New_York_City,117,Manhattan Island is linked to New York City's ...
16865,Roman_Republic,28,Caesar was now the primary figure of the Roman...
16273,Modern_history,79,"The Holocaust (which roughly means ""burnt whol..."


In [15]:
parsed_df.to_csv('parsed.tsv', sep='\t', index=False)

Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [16]:
parsed_copy = parsed_df.copy()
# parsed_copy['context'] = parsed_copy['paragraph_context'].apply(lambda x: x.get('context'))
parsed_copy.head()

,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...
3,University_of_Notre_Dame,3,The College of Engineering was established in ...
4,University_of_Notre_Dame,4,All of Notre Dame's undergraduate students are...


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import json

# extract context from the paragraph_context's dictionary using 'context' key
paragraph_vectors = parsed_df.copy()
# paragraph_vectors['context'] = paragraph_vectors['paragraph_context'].apply(lambda x: x.get('context')) 

# tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=1024, norm='l2', smooth_idf=True)
tfidf_vectorizer = TfidfVectorizer(max_features=1024, norm='l2', smooth_idf=True)     
ctx_tfidf = tfidf_vectorizer.fit_transform(paragraph_vectors['paragraph_context'])
round_vec = np.round(ctx_tfidf.toarray(), decimals=5)        # rounding to 5 decimals to save memory

paragraph_vec_df = pd.DataFrame({
    'document_title': paragraph_vectors['document_title'],
    'paragraph_index': paragraph_vectors['paragraph_index'],
    'paragraph_vector_json': [json.dumps(vec.tolist()) for vec in round_vec]
})

paragraph_vec_df.sample(5)

,document_title,paragraph_index,paragraph_vector_json
971,Portugal,49,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.13005, 0..."
3815,Bern,15,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.089..."
8344,Hellenistic_period,19,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3252,Web_browser,5,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
13978,Karl_Popper,35,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [18]:
round_vec.shape

(18896, 1024)

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [19]:
paragraph_vec_df.to_csv('paragraph-vectors.tsv.gz', sep='\t', index=False)

Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [20]:
questions = pd.read_csv('questions.tsv', sep='\t')
questions.head()

,question_id,question
0,1,What was the goal of the abuse of region project?
1,4,How many satellites in the Beidou-1 constellat...
2,7,When did Beyoncé receive ten nominations for ...
3,10,"With which goddess did Sulla, Pompey, and Juli..."
4,13,What area is considered to have a desert clima...


In [21]:
# encode questions using same tfidvectorizer with normalization and rounding
quest_tfidf      = tfidf_vectorizer.transform(questions['question'])
quest_round_vec  = np.round(quest_tfidf.toarray(), decimals=5)        # rounding to 5 decimals to save memory

question_vec_df = pd.DataFrame({
    'question_id': questions['question_id'],
    'question_vector_json': [json.dumps(vec.tolist()) for vec in quest_round_vec]
})

question_vec_df.sample(5)

,question_id,question_vector_json
92,277,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
15,46,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
8,25,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9,28,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
91,274,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [22]:
is_zero_vec = np.all(quest_round_vec == 0, axis=1)
print(f'There are {is_zero_vec.sum()} zero vectors')

zero_question_ids = questions.loc[is_zero_vec, 'question_id'].tolist()
print(f'Question IDs with zero vectors:')
print(zero_question_ids)

print(f"\n{questions.loc[is_zero_vec, 'question'].values}")

There are 0 zero vectors
Question IDs with zero vectors:
[]

<ArrowStringArray>
[]
Length: 0, dtype: str


Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [23]:
question_vec_df.to_csv('question-vectors.tsv', sep='\t', index=False)

Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [24]:
from sklearn.neighbors import NearestNeighbors

# convert vectors to numpy arrays
paragraph_vec = np.array(paragraph_vec_df['paragraph_vector_json'].apply(json.loads).tolist())
question_vec  = np.array(question_vec_df['question_vector_json'].apply(json.loads).tolist())

# 5-nearest neighbors
model = NearestNeighbors(n_neighbors=5, metric='euclidean')
model.fit(paragraph_vec)

# get the distances and indices of the top 5 paragraphs for each question
distances, indices = model.kneighbors(question_vec)

print('Top 5 paragraph indices for the first question:')
print(indices[0])
print('Euclidean Distances:')
print(distances[0])

Top 5 paragraph indices for the first question:
[17565  1682 15050  8514 13052]
Euclidean Distances:
[0.96707951 1.06718847 1.06821915 1.08058735 1.09104299]


In [25]:
matches = []

# loop through each question and its top 5 paragraphs
for q_idx, top_idx in enumerate(indices):

    # get question ID 
    q_id = question_vec_df.iloc[q_idx]['question_id']

    # loop through each of the top 5 paragraphs and add match data
    for rank, p_idx in enumerate(top_idx, start=1):

        # get document title and paragraph index
        title = paragraph_vec_df.iloc[p_idx]['document_title']
        paragraph_idx = paragraph_vec_df.iloc[p_idx]['paragraph_index']

        matches.append({
            'question_id': q_id,
            'question_rank': rank,
            'document_title': title,
            'paragraph_index': paragraph_idx
        })

matches_df = pd.DataFrame(matches)
matches_df.head(10)

,question_id,question_rank,document_title,paragraph_index
0,1,1,Tuvalu,49
1,1,2,Genome,8
2,1,3,Rajasthan,9
3,1,4,Bill_%26_Melinda_Gates_Foundation,6
4,1,5,Tibet,10
5,4,1,Dog,27
6,4,2,Multiracial_American,17
7,4,3,The_Blitz,10
8,4,4,Rule_of_law,3
9,4,5,Classical_music,5


Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [26]:
matches_df.to_csv('question-matches.tsv', sep='\t', index=False)

Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


In [27]:
matches_sorted = matches_df.sort_values(by='question_id')

# get the first 5 question IDs in ascending order
first_5_question_ids = matches_sorted['question_id'].unique()[:5]

# filter the matches by the first 5 questions
first_5_questions = matches_sorted[matches_sorted['question_id'].isin(first_5_question_ids)]

# filter to worst matches (rank 5)
worst_matches = first_5_questions[first_5_questions['question_rank'] == 5][['question_id', 'document_title', 'paragraph_index']]
worst_matches

,question_id,document_title,paragraph_index
4,1,Tibet,10
9,4,Classical_music,5
14,7,High-definition_television,11
19,10,Symbiosis,8
24,13,Southeast_Asia,9


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [28]:
worst_matches.to_csv('worst-paragraphs.tsv', sep='\t', index=False)

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

none